# Latent Failure Forecasting PoC on Google Colab (SWE-bench Lite)

Probes whether a trajectory's eventual success/failure becomes more decodable from the residual stream as the agent nears its own conclusion. Streams pre-generated **SWE-bench** agent trajectories (`nebius/swe-agent-trajectories`), replays each through Llama-3.2-1B on GPU caching residual activations at step boundaries, then trains logistic-regression probes (layer x relative-position). No live agent is run; this replaces the earlier (too-short) HotpotQA version.

1. `Runtime > Change runtime type > T4 GPU > Save`, then **`Runtime > Restart session`**
2. Run cells **in order** (clone → install → **restart** → GPU check → verify deps → …). Do not import `torch` before the install cell finishes.
3. **After the install cell, `Runtime > Restart session` again** before GPU check. Never `reload(torch)` in the same session after pip — it crashes the kernel.

**About `^C` right after `Loading weights`:** runtime OOM during model load — model loads fp16 to avoid it.

**About `n_ctx` / `reason` / `nms` errors:** re-run install cell only on a **fresh restart** (no prior `import torch` in this session), then restart again before verify-deps.

In [ ]:
import os
if os.path.basename(os.getcwd()) != 'algoverse-PoC':
    if not os.path.isdir('algoverse-PoC'):
        !git clone https://github.com/abdelmagid07/algoverse-PoC.git
    %cd algoverse-PoC
!git pull --ff-only

In [ ]:
# Install WITHOUT importing torch in this kernel (importing torch before pip,
# then reinstalling/reloading it, corrupts the runtime). Read Colab torch in a
# subprocess only. Install TL with --no-deps so pip does not touch torch.
import subprocess, sys

def _torch_tuple(ver: str) -> tuple[int, ...]:
    return tuple(int(x) for x in ver.split('+')[0].split('.')[:3])

out = subprocess.check_output([
    sys.executable, '-c',
    'import torch; print(torch.__version__); print(torch.version.cuda or "128")',
], text=True).strip().splitlines()
colab_torch, cuda = out[0], out[1]
if _torch_tuple(colab_torch) < (2, 7, 0):
    raise RuntimeError(f'Colab torch {colab_torch} < 2.7; factory-reset runtime.')
print(f'Colab torch {colab_torch} (subprocess check — kernel has not imported torch)')

# TL deps except torch/torchvision (pip must not replace Colab's CUDA torch).
_TL_DEPS = (
    'accelerate beartype better-abc einops fancy-einsum jaxtyping rich '
    'sentencepiece transformers-stream-generator typeguard typing-extensions '
    'tqdm huggingface-hub packaging protobuf pandas wandb'
)

!pip install -q --no-deps "transformer_lens>=3.4.0"
!pip install -q "transformers>=5.4.0"
!pip install -q {_TL_DEPS}
!pip install -q datasets scipy scikit-learn matplotlib numpy
!pip uninstall -q -y torchvision || true

print('\nInstall done. Do NOT import torch/transformers here.')
print('>>> Runtime > Restart session, then run GPU-check + Verify-deps cells.')

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'CUDA not visible. Set runtime to T4 GPU, then Runtime > Restart session, then re-run.'
)
print(torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
import psutil
print(f'System RAM: {psutil.virtual_memory().total / 1e9:.1f} GB')
print(f'torch={torch.__version__}')

In [ ]:
# Run AFTER restart — first transformers import in this session.
import inspect
import torch
import transformers
import transformer_lens

if 'reason' not in inspect.signature(torch.compiler.disable).parameters:
    raise RuntimeError(
        f'torch {torch.__version__} missing compiler.disable(reason=). '
        f'Factory-reset runtime, run clone+install (no GPU check before install), restart, retry.'
    )
print(f'torch={torch.__version__}  transformers={transformers.__version__}  '
      f'transformer_lens={getattr(transformer_lens, "__version__", "?")}')
print('Imports OK — continue to HF login.')

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(getpass('HF token: '))

In [ ]:
# Smoke test: load the model in-process (fp16). Confirms GPU + that loading does not OOM.
import os
os.environ['VERITAS_DEVICE'] = 'cuda'

from interp.activation_cache import load_model, log_device_choice
log_device_choice()
model = load_model()
print('Loaded:', model.cfg.model_name, '| dtype:', next(model.parameters()).dtype,
      '| device:', next(model.parameters()).device)
print(f'GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# Full pipeline, in-process (~15-30 min). Wait for '=== Done ==='.
# Phase 1 streams SWE-bench trajectories and replays each through the model
# (one forward pass capped at MAX_CONTEXT_TOKENS), caching only step-boundary
# activations. If the runtime crashes/disconnects, just re-run this cell - it
# resumes from results/trajectories.json checkpoints.
import run_pipeline
run_pipeline.main()

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

for p in ['results/probe_results.json', 'results/accuracy_by_position.png', 'results/poc_summary.md']:
    assert Path(p).exists(), f'Missing {p} - run the pipeline cell to completion'

for img in ['results/accuracy_by_position.png', 'results/accuracy_by_layer.png', 'results/probe_heatmap.png']:
    display(Image(img))
display(Markdown(open('results/poc_summary.md').read()))